# Topic 24 - Chunking

Companion notebook for `chunking.py`. The single highest-leverage quality decision in RAG: chunk the sample document three different ways (fixed-size, sentence-aware, markdown structure-aware) and read the granularity trade-off off one table - then see what overlap actually buys and costs, and how the LLM-side token budget reverse-engineers your chunk size.

## Learning objectives

- see the topic's mechanics running offline, cell by cell
- connect each demo cell to the pytest suite that pins it
- walk away with numbers you can quote from memory

## 0. Import the core

In [1]:
from chunking import (
    SAMPLE_DOCUMENT, balanced_fences, estimate_token_count, fixed_size_chunks,
    markdown_chunks, mid_sentence_cut_count, mid_table_cut_count,
    overlap_text, paragraph_chunks, report, sentence_chunks, token_budget_chunks,
)
doc = SAMPLE_DOCUMENT
print(f"document = {len(doc)} chars, ~{estimate_token_count(doc)} tokens (chars/4 estimate)")


document = 1381 chars, ~346 tokens (chars/4 estimate)


## 1. The sample document and its traps

Headings, a table, a fenced code block, an abbreviation ('Dr.'), an exclamation run, and one long paragraph - exactly the joints that punish naive windowing.

In [2]:
# skeleton view: structural lines marked '!'
for line in doc.splitlines():
    if line.strip():
        marker = "!" if (line.startswith("#") or line.startswith("|")) else " "
        print(f"{marker} {line[:72]}")


! # Return Policy
  We accept returns within 30 days. Refunds go back to the original paymen
  Sale items!!! Are final sale only. Dr. Smith recommends checking the man
! ## Exclusions
  Electronics are the exception - see the table below.
! | Product | Window | Notes |
! |---------|--------|-------|
! | Laptops | 14 days | Must include charger |
! | Phones | 7 days | Activation locks must be off |
! | Headphones | 30 days | Hygiene seal intact |
  Wireless headphones have a special note: hygiene seals matter enormously
! ## Shipping Damages
  ```python
  def is_returnable(item, days_since_purchase):
      if item.from_sale and days_since_purchase > 0:
          return False  # sale items are final
      return days_since_purchase <= item.return_window
  ```
  Photograph the box before opening. Carriers deny claims without photos, 


## 2. Strategy A: fixed-size windows (the baseline floor)

In [3]:
fixed = fixed_size_chunks(doc, size=200)
rep = report(fixed, doc)
print(f"chunks: {rep.chunk_count}   avg chars: {rep.avg_chars:.0f}   max: {rep.max_chars}")
print(f"mid-sentence cuts: {rep.mid_sentence_cuts}")
print(f"chunks touching a table edge: {mid_table_cut_count(fixed)}")
bad = [c for c in fixed if not balanced_fences(c.text) and "```" in c.text]
print(f"chunks that slice a code fence: {len(bad)}")
print("\nsample casualty (one window):")
print(repr(fixed[5].text))


chunks: 7   avg chars: 197   max: 200
mid-sentence cuts: 6
chunks touching a table edge: 2
chunks that slice a code fence: 2

sample casualty (one window):
'if item.from_sale and days_since_purchase > 0:\n        return False  # sale items are final\n    return days_since_purchase <= item.return_window\n```\n\nPhotograph the box before opening. Carriers deny c'


## 3. Strategy B: sentence-aware packing

Whole sentences until full; zero mid-sentence cuts; hard-splits are flagged in metadata, never silent.

In [4]:
sents = sentence_chunks(doc, max_chars=300)
rep = report(sents, doc)
print(f"chunks: {rep.chunk_count}   avg chars: {rep.avg_chars:.0f}   max: {rep.max_chars}")
print(f"mid-sentence cuts: {rep.mid_sentence_cuts}")
hard = [c for c in sents if c.metadata.get("hard_split")]
print(f"hard splits: {len(hard)}")
print("\nsample chunk text (one clean chunk):")
print(sents[2].text[:200] + " ...")


chunks: 7   avg chars: 196   max: 275
mid-sentence cuts: 0
hard splits: 0

sample chunk text (one clean chunk):
## Exclusions

Electronics are the exception - see the table below. ...


## 4. Strategy C: markdown structure-aware

Heading hierarchy rides in metadata (`header_path`); fences never split (even when chunks are tiny); the table survives whole under its heading.

In [5]:
mds = markdown_chunks(doc)
print(f"chunks: {len(mds)}")
for c in mds:
    fence_ok = balanced_fences(c.text)
    table_ok = "| Headphones |" not in c.text or "| Product |" in c.text
    print(f"  {c.id}: path={c.metadata['header_path']!r}  fences_ok={fence_ok}  table_intact={table_ok}  chars={len(c.text)}")


chunks: 3
  md-1: path='Return Policy'  fences_ok=True  table_intact=True  chars=411
  md-2: path='Return Policy > Exclusions'  fences_ok=True  table_intact=True  chars=508
  md-3: path='Return Policy > Shipping Damages'  fences_ok=True  table_intact=True  chars=462


## 5. The trade-off table (the done-when artifact)

In [6]:
print(f"{'strategy':10s} {'chunks':>7s} {'avg':>5s} {'max':>5s} {'bad cuts':>9s}")
print("-" * 44)
for chunks in (fixed, sents, mds):
    r = report(chunks, doc)
    print(f"{r.strategy:10s} {r.chunk_count:7d} {r.avg_chars:5.0f} {r.max_chars:5d} {r.mid_sentence_cuts:9d}")
print()
print("fine-grained      -> many chunks -> broken sentences")
print("structure-aware   -> few chunks   -> clean cuts, bigger context per hit")


strategy    chunks   avg   max  bad cuts
--------------------------------------------
fixed            7   197   200         6
sentence         7   196   275         0
markdown         3   460   508         0

fine-grained      -> many chunks -> broken sentences
structure-aware   -> few chunks   -> clean cuts, bigger context per hit


## 6. Overlap: what it buys, what it costs

Overlap raises recall (the true sentence survives somewhere intact more often) and adds a duplicate tax to retrieval - and does nothing for tables or code fences.

In [7]:
fixed_overlap = fixed_size_chunks(doc, size=200, overlap=50)
print(f"no overlap : {len(fixed)} chunks")
print(f"50 overlap : {len(fixed_overlap)} chunks")
shared = sum(len(overlap_text(p, n)) for p, n in zip(fixed_overlap, fixed_overlap[1:]))
print(f"duplicated characters re-embedded and re-ranked: {shared}")


no overlap : 7 chunks
50 overlap : 9 chunks
duplicated characters re-embedded and re-ranked: 400


## 7. Token budgets: size from the LLM backward

budget = (context window - reserved answer - reserved instructions) / n. Estimate tokens as chars/4; swap in a real counter in production.

In [8]:
print(f"{'budget':>6s} {'chunks':>7s} {'max tokens/chunk':>17s}")
for budget in (400, 200, 100, 50):
    packs = token_budget_chunks(doc, budget_tokens=budget)
    max_tok = max(c.metadata["tokens"] for c in packs)
    print(f"{budget:6d} {len(packs):7d} {max_tok:17d}")


budget  chunks  max tokens/chunk
   400       1               344
   200       2               188
   100       5                89
    50       9                50


## 8. Run the tests

In [9]:
import subprocess
import sys

result = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(result.stdout[-1500:])
assert "failed" not in result.stdout, "pytest failures!"
print("\noffline tests passed")


..............................                                           [100%]
30 passed in 0.03s


offline tests passed


## 9. What to notice

- fixed windows cut the table, the fence, and 6 sentences; overlap pays in duplicates
- sentence-aware fixes cuts; structure-aware fixes context proactively (header_path)
- the trade-off table IS the deliverable: chunk count up, cuts down, granularity named
- token budget comes from prompt arithmetic, not from habit
- re-chunking is a schema migration: hash content, re-embed, upsert (Topic 26)